# Binance Data Downloader

### Objective:
- Download and save historical price data from the Binance cryptocurrency exchange as CSV files.

In [29]:
# Collecting all imports in one place
import pandas as pd
import requests
import time
import random
import re
import zipfile
import os 

## Function for data downloading

In [12]:
#Function for downloading data from Binance website
def download_file(market_type='spot',      #Options: spot, futures/cm, futures/um, option.
                  time_aggregated='daily', #For spot, futures: daily, monthly. For option: daily.
                  data_type='klines',      #For spot: klines, aggTrades, trades. For futures: monthly: klines, aggTrades, trades, bookTicker, fundingRate, indexPriceKlines, markPriceKlines, premiumIndexKlines; daily: ..., liquidationSnapshot, metrics, bookDepth. For option: BVOLIndex, EOHSummary.
                  timeframe='/',           #For klines: /1s/, /1m/, /3m/, /5m/, /15m/, /30m/, /1h/, /2h/, /4h/, /6h/, /8h/, /12h/, /1d/, /3d/, /1w/, /1mo/. For everything else: /.
                  date='2024-01-01',       #Format for monthly: 2020-08, for daily: 2020-08-08
                  ticker='BTCUSDT'):
    
    #Format the final URL string if the data type is candlesticks
    if 'klines' in data_type.lower():
        url = f"https://data.binance.vision/data/{market_type}/{time_aggregated}/{data_type}/{ticker}{timeframe}{ticker}{re.sub(r'/', '-', timeframe)}{date}.zip"
    #In all other cases
    else:
        url = f"https://data.binance.vision/data/{market_type}/{time_aggregated}/{data_type}/{ticker}/{ticker}-{data_type}-{date}.zip"
    print(url)
    
    #Directory for saving the file
    directory = f"/Users/danielschollenberg/Documents/Трейдинг/Данные/binance/{market_type}/{data_type}/{ticker}"
    #Create directory if it does not exist
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory {directory} created.")
        
    #Bypassing potential blocking:
    #Add random delay from 0 to 2 seconds before request
    delay = random.uniform(0, 2)
    print(f"Delay before request: {delay:.2f} seconds")
    time.sleep(delay)
    #List of User-Agent headers
    user_agents = ['Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
                   'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.0.1 Safari/605.1.15',
                   'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:89.0) Gecko/20100101 Firefox/89.0',
                   'Mozilla/5.0 (iPhone; CPU iPhone OS 14_6 like Mac OS X) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/14.1.1 Mobile/15E148 Safari/604.1',
                   'Mozilla/5.0 (Linux; Android 10; SM-G975F) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/89.0.4389.105 Mobile Safari/537.36']
    #Select random User-Agent header
    headers = {'User-Agent': random.choice(user_agents)}
    
    #Create request to get data in streaming mode
    response = requests.get(url, headers=headers, stream=True)
    #If connection successful (status 200) download and save file
    if response.status_code == 200:
        filename = directory + f"/{ticker}_{date}.zip"
        with open(filename, 'wb') as f:
            #Write all content directly to file
            f.write(response.content)  
        print(f"File {filename} successfully downloaded and saved")
    else:
        print(f"Error downloading file: status {response.status_code}, URL: {url}")
        
    #Unpacks zip file into specified directory  
    def unpack_zip(zip_path, extract_to):
        try:
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_to)
            #Delete zip file after extraction
            os.remove(zip_path)
            print(f"Archive {zip_path} deleted after extraction.")
        except zipfile.BadZipFile:
            print("Error during extraction: file is not a zip archive")
    try:
        
        #Unpack file
        unpack_zip(filename, os.path.dirname(filename))
        print(f"File {filename} successfully extracted.")
    except:
        print('Error during extraction')

In [14]:
# Example usage
download_file(market_type='futures/um', data_type='trades', time_aggregated='daily', date='2024-01-01')

https://data.binance.vision/data/futures/um/daily/trades/BTCUSDT/BTCUSDT-trades-2024-01-01.zip
Directory /Users/danielschollenberg/Documents/Трейдинг/Данные/binance/futures/um/trades/BTCUSDT created.
Delay before request: 0.13 seconds
File /Users/danielschollenberg/Documents/Трейдинг/Данные/binance/futures/um/trades/BTCUSDT/BTCUSDT_2024-01-01.zip successfully downloaded and saved
Archive /Users/danielschollenberg/Documents/Трейдинг/Данные/binance/futures/um/trades/BTCUSDT/BTCUSDT_2024-01-01.zip deleted after extraction.
File /Users/danielschollenberg/Documents/Трейдинг/Данные/binance/futures/um/trades/BTCUSDT/BTCUSDT_2024-01-01.zip successfully extracted.
